# Why sqlglot? (Part 4)

Our agent generates SQL with an LLM. Before we run anything against PostgreSQL, we must check that the query is safe.

**sqlglot** parses SQL into an abstract syntax tree (AST): a structured object we can inspect in Python. That is much more reliable than treating SQL as plain text.

This notebook shows four small examples that mirror what `db/validate_sql.py` does.

In [ ]:
import sqlglot
from sqlglot import exp

### 1. What kind of statement is this?

We only allow read-only `SELECT` queries. sqlglot tells us the **statement type** after parsing — not just whether a dangerous word appears in the text.

In [ ]:
queries = [
    "SELECT product_id FROM products",
    "INSERT INTO products VALUES (1)",
    "COPY products TO STDOUT",
]

for sql in queries:
    tree = sqlglot.parse_one(sql, dialect="postgres")
    is_select = isinstance(tree, exp.Select)
    print(f"{type(tree).__name__:8} | SELECT allowed? {is_select} | {sql}")

`COPY` is dangerous but does not contain words like `insert` or `delete`. Parsing catches it because the AST node is `Copy`, not `Select`.

### 2. Which tables does the query touch?

We allow only specific Northwind tables. sqlglot walks the AST and collects every `Table` node — including tables inside `JOIN`s.

In [ ]:
sql = """
SELECT p.product_name, c.category_name
FROM products p
JOIN categories c ON p.category_id = c.category_id
"""

tree = sqlglot.parse_one(sql, dialect="postgres")
tables = {t.name.lower() for t in tree.find_all(exp.Table)}
print("Tables found:", sorted(tables))

### 3. Does the query have a LIMIT?

We cap result size. With the AST we can look for a `Limit` node instead of guessing with string checks.

In [ ]:
for sql in [
    "SELECT * FROM products",
    "SELECT * FROM products LIMIT 10",
]:
    tree = sqlglot.parse_one(sql, dialect="postgres")
    limit = tree.find(exp.Limit)
    print(f"LIMIT node: {limit!r:20} | {sql.strip()}")

If `limit` is `None`, our validator appends `LIMIT 100` before execution.

### 4. Why not only search for forbidden words?

A simple substring check on the raw SQL string is fast, but imprecise. sqlglot understands SQL structure.

In [ ]:
FORBIDDEN_KEYWORDS = {"create", "delete", "drop"}

safe_sql = "SELECT created_at FROM products"
bad_sql = "DELETE FROM products"


def blocked_by_substring(sql: str) -> bool:
    lowered = sql.lower()
    return any(keyword in lowered for keyword in FORBIDDEN_KEYWORDS)


def blocked_by_ast(sql: str) -> bool:
    tree = sqlglot.parse_one(sql, dialect="postgres")
    return not isinstance(tree, exp.Select)


for label, sql in [("safe column name", safe_sql), ("real DELETE", bad_sql)]:
    print(
        f"{label:18} | substring: {blocked_by_substring(sql):5} | AST: {blocked_by_ast(sql):5} | {sql}"
    )

The column `created_at` contains the letters `create`, so the substring check gives a **false positive**. The AST correctly sees a harmless `SELECT`.

In the project we use **both** checks: forbidden keywords as a quick first filter, then sqlglot parsing as the authoritative gate. See `db/validate_sql.py`.

### Bonus: invalid SQL fails at parse time

If the LLM returns broken SQL, sqlglot raises a parse error before we touch the database.

In [ ]:
broken_sql = "SELECT FROM products WHERE"

try:
    sqlglot.parse_one(broken_sql, dialect="postgres")
except sqlglot.errors.ParseError as exc:
    print("Parse rejected:", exc)